# 04 – Run All Experiments

Single end-to-end runner that produces every CSV the results-visualization notebook consumes. Each step shells out to a `scripts/*.py` CLI (so pipeline state lives in its own subprocess and VRAM is released between stages), streams stdout live, and skips work when the expected output already exists. Flip `FORCE = True` in the config cell to rerun a cell anyway.

Execution order (matters — downstream stages depend on the production selector checkpoint, so selector training runs first):
1. Selector training data — `train_selector.py` × detectors, collects `TRAIN_SELECTOR_TARGET_DETECTIONS` oracle-labeled rows per detector.
2. **Retrain production selector on `TRAIN_SELECTOR_MAIN_SIZE` samples** — inline Python, overwrites the checkpoint so the main AED-XAI eval uses a 500-sample model (our chosen operating point from the ablation sweep).
3. Selector training-size ablation — `ablation_selector_size.py` sweeps `SIZES × SEEDS` (default `[50, 100, 200, 500, 1000]`) on each detector's CSV, tagging rows with `--detector` so Figure 3b can plot one line per detector.
4. Fixed-method baselines — `run_baseline.py` × {gradcam, gcame, dclose, lime} × detectors, on `NUM_IMAGES = 3000`.
5. AED-XAI pipeline — `run_experiments.py` × detectors, on 3000 images, using the 500-sample production checkpoint.
6. Cross-domain evaluation — `run_cross_domain.py`.
7. Detector generalization comparison — `compare_detectors.py`.
8. Figure generation + `main_results.csv` cache — `generate_figures.py`.
9. Status check — OK/MISSING for every expected artifact.
10. Preview — loads each produced CSV and shows `.head()` inline so you can sanity-check before running `05_results_visualization.ipynb`.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
CHECKPOINTS_DIR = PROJECT_ROOT / "data" / "checkpoints"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- knobs ---
FORCE = False                            # True -> rerun even if output CSV already exists
NUM_IMAGES = 3000                        # images per main experiment / baseline
NUM_XDOMAIN_IMAGES = 200                 # images per domain for cross-domain
TRAIN_SELECTOR_MAX_IMAGES = 1000         # images scanned during oracle labeling
TRAIN_SELECTOR_TARGET_DETECTIONS = 2000  # total detections to collect for selector training CSV
TRAIN_SELECTOR_MAIN_SIZE = 500           # subsample size used for the PRODUCTION selector checkpoint
DETECTORS = ["yolox-s", "fasterrcnn_resnet50_fpn_v2"]
BASELINE_METHODS = ["gradcam", "gcame", "dclose", "lime"]
XDOMAIN_METHODS = "aedxai,gradcam,gcame,dclose,lime"
SIZES = "50,100,200,500,1000"            # selector training-size ablation sweep
SEEDS = "42,123,456"

print(f"PROJECT_ROOT                    = {PROJECT_ROOT}")
print(f"RESULTS_DIR                     = {RESULTS_DIR}")
print(f"CHECKPOINTS                     = {CHECKPOINTS_DIR}")
print(f"FORCE                           = {FORCE}")
print(f"NUM_IMAGES (COCO eval)          = {NUM_IMAGES}")
print(f"NUM_XDOMAIN_IMAGES             = {NUM_XDOMAIN_IMAGES}")
print(f"TRAIN_SELECTOR_TARGET_DETECTIONS= {TRAIN_SELECTOR_TARGET_DETECTIONS}")
print(f"TRAIN_SELECTOR_MAIN_SIZE        = {TRAIN_SELECTOR_MAIN_SIZE}")
print(f"SIZES (ablation)                = {SIZES}")
print(f"DETECTORS                       = {DETECTORS}")

In [ ]:
def run_script(argv: list[str], *, skip_if: Path | None = None, label: str | None = None) -> int:
    """Run a CLI script from PROJECT_ROOT, streaming stdout. Return exit code.

    If `skip_if` is provided and the path exists (and FORCE is False), the run is skipped.
    """
    tag = label or " ".join(argv[:2])
    if skip_if is not None and skip_if.exists() and not FORCE:
        print(f"[skip] {tag} — {skip_if.relative_to(PROJECT_ROOT)} exists (set FORCE=True to rerun)")
        return 0

    cmd = [sys.executable, *argv]
    print(f"\n$ {' '.join(cmd)}")
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    process = subprocess.Popen(
        cmd,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    if returncode != 0:
        print(f"[error] {tag} exited with {returncode}")
    else:
        print(f"[done] {tag}")
    return returncode


def baseline_output_path(detector: str, method: str) -> Path:
    """Mirror scripts/run_baseline.py filename convention."""
    folder = RESULTS_DIR / "baseline"
    if detector == "yolox-s":
        return folder / f"baseline_{method}_results.csv"
    return folder / f"baseline_{detector}_{method}_results.csv"


def aedxai_output_path(detector: str) -> Path:
    subdir = "aedxai" if detector == "yolox-s" else f"aedxai_{detector}"
    return RESULTS_DIR / subdir / "results.csv"


def preview_csv(path: Path, *, head_rows: int = 5) -> None:
    """Load a CSV and display its head + shape if it exists."""
    if not path.exists():
        print(f"[missing] {path.relative_to(PROJECT_ROOT)}")
        return
    df = pd.read_csv(path)
    print(f"\n>>> {path.relative_to(PROJECT_ROOT)}  shape={df.shape}")
    display(df.head(head_rows))

def selector_ablation_complete(summary_path: Path, detectors: list[str], sizes_csv: str) -> bool:
    """Return True only when cached selector ablation has every detector × size row."""
    if not summary_path.exists():
        return False
    try:
        summary = pd.read_csv(summary_path)
    except Exception as exc:
        print(f"[rerun] could not read {summary_path.relative_to(PROJECT_ROOT)}: {exc}")
        return False

    expected_sizes = {int(value.strip()) for value in sizes_csv.split(",") if value.strip()}
    required_columns = {"detector", "train_size"}
    if not required_columns.issubset(summary.columns):
        print("[rerun] selector ablation cache is missing detector/train_size columns")
        return False

    for detector in detectors:
        available = set(summary.loc[summary["detector"] == detector, "train_size"].astype(int))
        missing = sorted(expected_sizes - available)
        if missing:
            print(f"[rerun] selector ablation cache missing {detector} sizes: {missing}")
            return False
    return True


## 1. Selector training data

For each detector, `train_selector.py` runs every XAI method on the training images, picks the per-detection oracle, and writes:
- `results/xai_selector_training_data_<detector>.csv` — full labeled feature set (~`TRAIN_SELECTOR_TARGET_DETECTIONS` rows)
- `data/checkpoints/xai_selector_<detector>.pth` — MLP trained on all collected rows (this checkpoint gets OVERWRITTEN by step 2 with a `TRAIN_SELECTOR_MAIN_SIZE`-sample model).

The training CSV feeds both the ablation (step 3) and the retrain step below, so we want enough rows to cover the biggest ablation size (`SIZES` max = 1000) — hence the default target of 2000 detections. The actual training-size sweep is `50, 100, 200, 500, 1000`, while the production selector used by the main AED-XAI run is retrained on exactly 500 rows in the next step.

In [ ]:
for detector in DETECTORS:
    training_csv = RESULTS_DIR / f"xai_selector_training_data_{detector}.csv"
    run_script(
        [
            "scripts/train_selector.py",
            "--detector-model", detector,
            "--max-images", str(TRAIN_SELECTOR_MAX_IMAGES),
            "--target-detections", str(TRAIN_SELECTOR_TARGET_DETECTIONS),
        ],
        skip_if=training_csv,
        label=f"train_selector / {detector}",
    )

## 2. Retrain production selector on `TRAIN_SELECTOR_MAIN_SIZE` samples

`train_selector.py` trains its checkpoint on the whole collected CSV (typically ~2000 rows). For the paper, we report results at the 500-sample operating point of the ablation sweep — so after the full training CSV is built, we retrain the selector on exactly `TRAIN_SELECTOR_MAIN_SIZE` subsampled rows and overwrite `data/checkpoints/xai_selector_<detector>.pth`. All downstream eval steps (AED-XAI, cross-domain, detector compare) then load this 500-sample checkpoint.

The ablation script in step 3 reads the full CSV directly, so the ablation still has access to the 1000-sample data point.

In [ ]:
from src.xai_selector import XAISelector

for detector in DETECTORS:
    csv_path = RESULTS_DIR / f"xai_selector_training_data_{detector}.csv"
    ckpt_path = CHECKPOINTS_DIR / f"xai_selector_{detector}.pth"
    if not csv_path.exists():
        print(f"[skip] retrain / {detector} — missing {csv_path.relative_to(PROJECT_ROOT)}")
        continue

    full_df = pd.read_csv(csv_path)
    if len(full_df) < TRAIN_SELECTOR_MAIN_SIZE:
        raise ValueError(
            f"{csv_path.relative_to(PROJECT_ROOT)} has only {len(full_df)} rows; "
            f"need at least {TRAIN_SELECTOR_MAIN_SIZE} for the production selector."
        )
    subsample_size = TRAIN_SELECTOR_MAIN_SIZE
    subsample = full_df.sample(n=subsample_size, random_state=42).reset_index(drop=True)

    selector = XAISelector()
    metrics = selector._train_from_dataframe(  # noqa: SLF001 — internal retrain path
        dataframe=subsample,
        val_split=0.2,
        test_split=0.05,
        epochs=100,
        lr=0.001,
        batch_size=32,
    )
    selector.save_model(str(ckpt_path))
    print(
        f"[done] retrain / {detector} — {subsample_size} rows  "
        f"train_acc={metrics['train_acc']:.3f}  val_acc={metrics['val_acc']:.3f}  "
        f"-> {ckpt_path.relative_to(PROJECT_ROOT)}"
    )

## 3. Selector training-size ablation

Sweeps `SIZES × SEEDS` on each detector's training CSV and aggregates into `results/ablation_selector_size_summary.csv`. The `--detector` flag stamps a `detector` column on every row so notebook 05's Figure 3b can draw one line per detector family. The second detector passes `--append` to accumulate rows instead of overwriting.

In [ ]:
ablation_summary = RESULTS_DIR / "ablation_selector_size_summary.csv"
if selector_ablation_complete(ablation_summary, DETECTORS, SIZES) and not FORCE:
    print(
        f"[skip] selector-size ablation — {ablation_summary.relative_to(PROJECT_ROOT)} "
        "already contains every detector × size row (set FORCE=True to rerun)"
    )
else:
    for index, detector in enumerate(DETECTORS):
        training_csv = f"results/xai_selector_training_data_{detector}.csv"
        argv = [
            "scripts/ablation_selector_size.py",
            "--training-csv", training_csv,
            "--detector", detector,
            "--sizes", SIZES,
            "--seeds", SEEDS,
        ]
        if index > 0:
            argv.append("--append")
        run_script(
            argv,
            skip_if=None,
            label=f"ablation_selector_size / {detector}",
        )


## 4. Fixed-method baselines

Runs `gradcam`, `gcame`, `dclose`, `lime` as non-adaptive baselines for each detector on `NUM_IMAGES` images. Outputs land under `results/baseline/`.

In [ ]:
for detector in DETECTORS:
    for method in BASELINE_METHODS:
        expected = baseline_output_path(detector, method)
        run_script(
            [
                "scripts/run_baseline.py",
                "--xai-method", method,
                "--detector-model", detector,
                "--num-images", str(NUM_IMAGES),
                "--output", "results/baseline",
            ],
            skip_if=expected,
            label=f"baseline {method} / {detector}",
        )

## 5. AED-XAI pipeline (adaptive, feedback-loop)

Runs the full pipeline — detector → VLM → adaptive XAI selector → feedback loop — for each detector family on `NUM_IMAGES` images. The pipeline auto-loads `data/checkpoints/xai_selector_<detector>.pth` (the 500-sample checkpoint produced in step 2). Outputs land under `results/aedxai[_<detector>]/results.csv`.

In [ ]:
for detector in DETECTORS:
    output_dir = "results/aedxai" if detector == "yolox-s" else f"results/aedxai_{detector}"
    expected = aedxai_output_path(detector)
    run_script(
        [
            "scripts/run_experiments.py",
            "--detector-model", detector,
            "--num-images", str(NUM_IMAGES),
            "--output", output_dir,
        ],
        skip_if=expected,
        label=f"aedxai / {detector}",
    )

## 6. Cross-domain evaluation

Runs AED-XAI and each fixed baseline on `NUM_XDOMAIN_IMAGES = 200` images per available domain (COCO / VOC / BDD100K / VisDrone / DOTA / OpenImages). Missing domain directories are skipped unless you pass `--strict`. Outputs: `results/cross_domain.csv`, `results/cross_domain_per_image.csv`, `results/cross_domain_summary.csv`.

In [ ]:
cross_domain_csv = RESULTS_DIR / "cross_domain.csv"
run_script(
    [
        "scripts/run_cross_domain.py",
        "--num-images", str(NUM_XDOMAIN_IMAGES),
        "--methods", XDOMAIN_METHODS,
        "--output-dir", "results",
    ],
    skip_if=cross_domain_csv,
    label="cross-domain",
)

## 7. Detector generalization comparison

Runs the full AED-XAI pipeline on the same image set with both detector families so notebook 05's Figure 3c (`fig3c_detector_comparison`) can render. Output: `results/compare_detectors_per_image.csv` + `results/compare_detectors_summary.csv`.

In [ ]:
compare_detectors_csv = RESULTS_DIR / "compare_detectors_per_image.csv"
run_script(
    [
        "scripts/compare_detectors.py",
        "--max-images", str(NUM_IMAGES),
        "--output-dir", "results",
    ],
    skip_if=compare_detectors_csv,
    label="compare_detectors",
)

## 8. Figure generation + `main_results.csv` cache

`generate_figures.py` stitches the per-method CSVs above into the unified schema notebook 05 reads (`method, pg, ebpg, oa, sparsity, composite, computation_time, image_id`) and caches `results/main_results.csv` as a side effect.

In [ ]:
main_results_csv = RESULTS_DIR / "main_results.csv"
run_script(
    [
        "scripts/generate_figures.py",
        "--results-root", "results",
    ],
    skip_if=main_results_csv,
    label="generate_figures",
)

## 9. Status check

Reports whether each artifact notebook 05 expects is present.

In [ ]:
expected = {
    "main_results.csv (required for notebook 05)": RESULTS_DIR / "main_results.csv",
    "cross_domain.csv": RESULTS_DIR / "cross_domain.csv",
    "compare_detectors_per_image.csv": RESULTS_DIR / "compare_detectors_per_image.csv",
    "ablation_selector_size_summary.csv": RESULTS_DIR / "ablation_selector_size_summary.csv",
    "ablation_selector_size.csv": RESULTS_DIR / "ablation_selector_size.csv",
}
for detector in DETECTORS:
    expected[f"xai_selector_training_data_{detector}.csv"] = (
        RESULTS_DIR / f"xai_selector_training_data_{detector}.csv"
    )
    expected[f"xai_selector_{detector}.pth (500-sample)"] = CHECKPOINTS_DIR / f"xai_selector_{detector}.pth"
    expected[f"aedxai results.csv ({detector})"] = aedxai_output_path(detector)
    for method in BASELINE_METHODS:
        expected[f"baseline {method} ({detector})"] = baseline_output_path(detector, method)

max_len = max(len(key) for key in expected)
for label, path in expected.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{label.ljust(max_len)}  {status:<8}  {path.relative_to(PROJECT_ROOT)}")

## 10. Preview produced CSVs

Sanity-check each artifact before moving to `05_results_visualization.ipynb`. Missing CSVs are reported inline so you can rerun the upstream cell.

In [ ]:
preview_targets: list[Path] = [
    RESULTS_DIR / "main_results.csv",
    RESULTS_DIR / "ablation_selector_size_summary.csv",
    RESULTS_DIR / "cross_domain_summary.csv",
    RESULTS_DIR / "compare_detectors_per_image.csv",
]
preview_targets.extend(aedxai_output_path(detector) for detector in DETECTORS)
preview_targets.extend(
    baseline_output_path(detector, method)
    for detector in DETECTORS
    for method in BASELINE_METHODS
)

for path in preview_targets:
    preview_csv(path)

print("\nNext: open notebooks/05_results_visualization.ipynb to render the paper figures.")